# Semiconductor Market Data Download

This notebook downloads a clean daily dataset for a semiconductor-focused pairs trading project.

It collects:
- a semiconductor stock universe
- benchmark ETFs for sector and market context
- adjusted price series for pair selection and backtesting
- volume data for basic liquidity checks

The outputs are saved as CSV files under `data/raw/`.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import yfinance as yf

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)

## Data to use

This starter universe mixes chip designers, memory, analog and RF names, and semiconductor equipment companies.
That gives you a much more sensible pairs universe than using all technology stocks.

In [2]:
ANALYSIS_START_DATE = "2023-01-01"
ANALYSIS_END_DATE = "2024-12-31"

TRADE_START_DATE = "2025-01-01"
TRADE_END_DATE = "2026-03-31"

SEMI_TICKERS = [
    "NVDA", "AMD", "AVGO", "QCOM", "TXN", "MU", "ADI", "MCHP", "NXPI", "ON",
    "MPWR", "INTC", "MRVL", "SWKS", "QRVO", "AMAT", "LRCX", "KLAC", "TER", "ASML",
]


BENCHMARK_TICKERS = ["SMH", "SOXX", "QQQ", "SPY"]


TICKER_COMPANIES = {
    "NVDA": "NVIDIA",
    "AMD": "Advanced Micro Devices",
    "AVGO": "Broadcom",
    "QCOM": "Qualcomm",
    "TXN": "Texas Instruments",
    "MU": "Micron Technology",
    "ADI": "Analog Devices",
    "MCHP": "Microchip Technology",
    "NXPI": "NXP Semiconductors",
    "ON": "ON Semiconductor",
    "MPWR": "Monolithic Power Systems",
    "INTC": "Intel",
    "MRVL": "Marvell Technology",
    "SWKS": "Skyworks Solutions",
    "QRVO": "Qorvo",
    "AMAT": "Applied Materials",
    "LRCX": "Lam Research",
    "KLAC": "KLA Corporation",
    "TER": "Teradyne",
    "ASML": "ASML Holding",
    "SMH": "VanEck Semiconductor ETF",
    "SOXX": "iShares Semiconductor ETF",
    "QQQ": "Invesco QQQ Trust",
    "SPY": "SPDR S&P 500 ETF Trust",
}

TICKER_CATEGORIES = {
    "NVDA": "Compute and AI",
    "AMD": "Compute and AI",
    "AVGO": "Compute and AI",
    "INTC": "Compute and AI",
    "MRVL": "Compute and AI",
    "MU": "Memory",
    "TXN": "Analog and Embedded",
    "ADI": "Analog and Embedded",
    "MCHP": "Analog and Embedded",
    "NXPI": "Analog and Embedded",
    "ON": "Analog and Embedded",
    "MPWR": "Analog and Embedded",
    "QCOM": "Wireless and RF",
    "SWKS": "Wireless and RF",
    "QRVO": "Wireless and RF",
    "AMAT": "Equipment and Manufacturing",
    "LRCX": "Equipment and Manufacturing",
    "KLAC": "Equipment and Manufacturing",
    "TER": "Equipment and Manufacturing",
    "ASML": "Equipment and Manufacturing",
    "SMH": "Benchmarks",
    "SOXX": "Benchmarks",
    "QQQ": "Benchmarks",
    "SPY": "Benchmarks",
}

ALL_TICKERS = SEMI_TICKERS + BENCHMARK_TICKERS


print(f"Downloading {len(SEMI_TICKERS)} semiconductor stocks and {len(BENCHMARK_TICKERS)} benchmarks")

## Download market data

In [3]:
raw = yf.download(
    tickers=ALL_TICKERS,
    start=ANALYSIS_START_DATE,
    end=TRADE_END_DATE,
    auto_adjust=True,
    progress=False,
    group_by="column",
    threads=True,
)



if raw.empty:
    raise ValueError("No data was downloaded. Check your internet connection and ticker list.")

raw.tail()

Price            Close                                                                                                               ...  \
Ticker             ADI        AMAT         AMD         ASML        AVGO       INTC         KLAC        LRCX       MCHP         MPWR  ...   
Date                                                                                                                                 ...   
2026-03-24  321.829987  373.989990  205.369995  1399.420044  318.290009  44.060001  1566.189941  238.839996  65.629997  1099.391968  ...   
2026-03-25  322.029999  369.339996  220.270004  1393.890015  318.809998  47.180000  1543.819946  233.449997  65.160004  1116.427979  ...   
2026-03-26  313.420013  338.549988  203.770004  1329.500000  309.420013  44.099998  1451.130005  211.619995  64.199997  1056.168457  ...   
2026-03-27  307.440002  337.170013  201.990005  1302.469971  300.679993  43.130001  1443.209961  211.410004  62.000000  1050.908936  ...   
2026-03-30  303.100006  323.119995  196.039993  1253.959961  293.410004  41.189999  1382.579956  199.929993  60.060001  1000.340027  ...   

Price        Volume                                                                                         
Ticker           ON      QCOM       QQQ     QRVO       SMH      SOXX        SPY     SWKS      TER      TXN  
Date                                                                                                        
2026-03-24  6831600   8409200  57750900  1385900   6177200   5287500   96457500  3115200  3207700  7015200  
2026-03-25  5537200   6669400  60475500   951300   7388700   5869000   90653800  2102600  3077100  4505800  
2026-03-26  7462000  11912500  81492100   910600  14806200  10172400   96494400  2586500  3017600  6314400  
2026-03-27  7136700   8964900  82702200  1148700  11811400   6334800  103649400  3443100  2164200  5792700  
2026-03-30  8398500  11555900  70602600   580800  12198200  13204800   99275900  3337000  3130100  6024700  

[5 rows x 120 columns]

## Prepare backtest inputs

In [4]:
close = raw["Close"].sort_index().dropna(axis=1, how="all")
volume = raw["Volume"].sort_index().dropna(axis=1, how="all")

close.columns.name = None
volume.columns.name = None

available_tickers = close.columns.tolist()
missing_tickers = sorted(set(ALL_TICKERS) - set(available_tickers))

print("Available tickers:", available_tickers)
print("Missing tickers:", missing_tickers)

display(close.head())

Available tickers: ['ADI', 'AMAT', 'AMD', 'ASML', 'AVGO', 'INTC', 'KLAC', 'LRCX', 'MCHP', 'MPWR', 'MRVL', 'MU', 'NVDA', 'NXPI', 'ON', 'QCOM', 'QQQ', 'QRVO', 'SMH', 'SOXX', 'SPY', 'SWKS', 'TER', 'TXN']
Missing tickers: []


,ADI,AMAT,AMD,ASML,AVGO,INTC,KLAC,LRCX,MCHP,MPWR,...,ON,QCOM,QQQ,QRVO,SMH,SOXX,SPY,SWKS,TER,TXN
Date,,,,,,,,,,,,,,,,,,,,,
2023-01-03,153.730560,94.039268,64.019997,534.022400,52.808151,25.775145,365.157776,40.103119,63.856056,334.519989,...,61.610001,99.506599,259.468750,89.139999,99.335976,112.213326,365.072052,81.330261,84.801590,148.055984
2023-01-04,157.004868,96.547508,64.660004,554.680908,53.453121,26.691212,368.726471,40.892982,65.417809,349.034180,...,62.200001,103.525887,260.704803,91.000000,101.861549,115.101768,367.890503,84.008339,86.272026,153.462601
2023-01-05,151.118713,95.196167,62.330002,549.326782,52.955078,26.575495,362.170990,40.370277,63.856056,338.528931,...,59.740002,101.548729,256.623688,90.290001,100.036430,113.160927,363.691589,83.055717,85.975998,151.430603
2023-01-06,156.635788,101.369522,63.959999,578.993103,56.142761,27.703699,385.561218,43.100925,67.247559,358.417755,...,62.470001,107.062424,263.706818,93.349998,104.411789,118.426582,372.031830,85.769714,90.041878,158.896439
2023-01-09,158.130981,103.527763,67.239998,603.752319,55.041718,28.262980,395.976318,43.794003,68.199402,374.034149,...,64.650002,106.384834,265.413879,95.209999,106.685783,120.608383,371.820923,86.704361,91.985992,160.275314


## Save outputs

In [5]:
# Because auto_adjust=True, `close` is already adjusted and can be used as adj_close.

close_analysis = close.loc[ANALYSIS_START_DATE:ANALYSIS_END_DATE]
close_trade = close.loc[TRADE_START_DATE:TRADE_END_DATE]
#TRADE_START_DATE = "2025-01-01"
#TRADE_END_DATE = "2026-03-31"
close_analysis.to_csv("Data/semiconductor_close_analysis.csv", index_label="date")
close_trade.to_csv("Data/semiconductor_close_trade.csv", index_label="date")

ticker_info = pd.DataFrame({
    "company": pd.Series(TICKER_COMPANIES),
    "category": pd.Series(TICKER_CATEGORIES),
})
ticker_info.index.name = "ticker"

ticker_info = ticker_info.sort_values(["category", "company"])

ticker_info.to_csv("Data/semiconductor_categories.csv")

#display(close_analysis)
#display(close_trade)

print("Saved files:")
print("- sc_close.csv")
print("- categories.csv")

Saved files:
- sc_close.csv
- categories.csv


## Suggested next step

Use `semiconductor_adj_close.csv` as the main input for pair selection and spread construction.
Use `semiconductor_volume.csv` if you want to add a simple liquidity filter before testing pairs.